<p align="left">
  <img src="https://img.shields.io/badge/Research%20Mode-ON-4cbb17?style=for-the-badge" alt="Research Mode">
</p>

# 02 · Data Exploration — ASAP CRN Learning Lab  
*A guided launchpad for your second ASAP-CRN workspace adventure.*

Welcome to the **ASAP-CRN Learning Lab Pilot Workshop Series!**  

This notebook walks you through the essentials of data inspection and preliminary analyses in **Verily Workbench**.

> **Tip:** Run each cell in order for the smoothest setup experience.  
> You can always come back later to experiment and make it your own.


**Before you start**
- Workspace resources mounted (`wb resource mount`)
- Access to the `pmdbs-sc-rnaseq-v3` dataset
- A VM with ≥32 GB RAM — section 2.4 loads the full gene matrix
- Runtime end to end: ~25 min, most of it in 1.4 and 2.4

**Prior notebook:** `01_getting_started.ipynb` · **Next:** `03_downstream_analysis.ipynb`

In [ ]:
# Builds the contents list from the notebook's own headers, so it stays
# correct when sections get added or renumbered.
import json, re
from pathlib import Path
from IPython.display import Markdown, display

NOTEBOOK = Path("02_data_exploration.ipynb")

def build_toc(path, max_level=3):
    cells = json.loads(path.read_text())["cells"]
    lines = []
    for cell in cells:
        if cell["cell_type"] != "markdown":
            continue
        for line in cell["source"]:
            m = re.match(r"^(#{2,%d})\s+(.+?)\s*$" % max_level, line)
            if m:
                indent = "  " * (len(m.group(1)) - 2)
                lines.append(f"{indent}- {m.group(2)}")
    return "\n".join(lines)

display(Markdown("### Table of Contents\n" + build_toc(NOTEBOOK)))

In [ ]:
import sys, subprocess, importlib, os, warnings, math, os, pandas as pd
from pathlib import Path
from typing import Sequence
import gc

def setup_environment():
    """Install any missing packages and return the libraries as a dict.

    Workbench VMs come with different package sets depending on the image,
    so we check rather than assume.
    """

    def import_or_install(pkg, name=None):
        try:
            return importlib.import_module(name or pkg)
        except ImportError:
            print(f"{pkg} not found. Installing...")
            subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)
            return importlib.import_module(name or pkg)

    env = {
        "pd": pd,
        "np": import_or_install("numpy"),
        "plt": import_or_install("matplotlib.pyplot", "matplotlib.pyplot"),
        "sns": import_or_install("seaborn"),
        "Image": import_or_install("PIL.Image", "PIL.Image"),
        "sc": import_or_install("scanpy")
    }
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 1200)
    
    print("Environment ready.")
    return env


In [ ]:
env = setup_environment()
pd, np, plt, sns, Image, sc = (
    env["pd"], env["np"], env["plt"], env["sns"],
    env["Image"], env["sc"]
)

**Key data objects used in this notebook**

- `adata` — full ASAP cohort-level AnnData (HVG-focused matrix).
- `full_adata` — full gene expression matrix (unfiltered genes) for the cohort, backed.
- `frontal_ad` — PFC-only subset with embeddings and metadata.
- `frontal_full_ad` — in-memory AnnData for PFC subset with **full gene matrix** + metadata and embeddings.


## 1. Workspace Setup

### 1.1 Set dataset paths
In this example, we are working with the **PMDBS single‑cell RNA‑seq cohort** dataset:

- **Workflow** → `pmdbs_sc_rnaseq`  
- **Team** → `cohort`  
- **Source** → `pmdbs`  
- **Type** → `sc-rnaseq`  

These components are combined to construct the bucket and dataset names.  
We then set the path to the **cohort analysis outputs** and preview the available files.


In [ ]:
#set general folder paths
HOME = Path.home()
WS_ROOT = HOME / "workspace"
DATA_DIR = WS_ROOT / "Data"
WS_FILES = WS_ROOT / "ws_files"

if not WS_ROOT.exists():
    print(f"{WS_ROOT} doesn't exist. We need to remount our resources")
    !wb resource mount    

print("Home directory:     ", HOME)
print("Workspace root:     ", WS_ROOT)
print("Data directory:     ", DATA_DIR)
print("ws_files directory: ", WS_FILES)

print("\nContents of workspace root:")
for p in WS_ROOT.glob("*"):
    print(" -", p.name, "/" if p.is_dir() else "")

In [ ]:
## Build and set path to desired dataset
DATASETS_PATH = WS_ROOT / "01_PMDBS" / "pmdbs-sc-rnaseq-v3"

workflow       = "pmdbs_sc_rnaseq"
dataset_team   = "cohort"
dataset_source = "pmdbs"
dataset_type   = "sc-rnaseq"


bucket_name  = f"{dataset_team}-{dataset_source}-{dataset_type}"
dataset_name = f"{dataset_team}-{dataset_source}-{dataset_type}"

dataset_path = DATASETS_PATH / bucket_name / workflow

local_data_path = WS_FILES / "pilot_workshop_files"

if not local_data_path.exists():
    raise FileNotFoundError(
        f"{local_data_path} doesn't exist! "
        "Check path or make sure to have run 01_getting_started.ipynb before this notebook"
    )
else: 
    print(f"Local data directory ready at: {local_data_path}\n")

print("Local Directory Contents:")
!ls  {local_data_path} 

cohort_analysis_path = dataset_path / "cohort_analysis"

!ls  {cohort_analysis_path} 

### 1.2 Define Metadata Path

Alongside the dataset, we also define a path to the **release metadata resources**.  
This folder contains tables describing samples, subjects, brain regions, and cinical pathology.  
Previewing the contents helps us confirm which metadata files are available for integration.


In [ ]:
#Define metadata folder path
ds_metadata_path = WS_ROOT / "release_resources/cohort-pmdbs-sc-rnaseq/metadata/release/v5.0.0"

#preview contents
!ls {ds_metadata_path} 

### 1.3 Create Local Directory

To keep our work organized, we create a local directory inside `ws_files` called `pilot_workshop_files`.  
This is where we’ll save any outputs (plots, tables, subsetted data) that we want to retain or share.  
If the directory doesn’t exist yet, we create it.


In [ ]:
# Define a local path for workshop files
local_data_path = WS_FILES / "pilot_workshop_files"

# Create the directory if it doesn't already exist
if not local_data_path.exists():
    local_data_path.mkdir(parents=True)

print(f"Local data directory ready at: {local_data_path}")

### 1.4. Copy Data Locally

We now bring in the curated dataset files:

- **`asap-cohort.final_metadata.csv`** → cell‑level metadata table
- **`asap-cohort.final.h5ad`** → full AnnData object containing HVG expression data and annotations  

We copy these files into our local `pilot_workshop_files` directory (if not already present) and load them into memory.

The metadata CSV is read into a Pandas dataframe, while the `.h5ad` file is loaded as an AnnData object in backed mode.


In [ ]:
# Define the expected local path
cell_metadata_local_path = local_data_path / f"asap-{dataset_team}.final_metadata.csv"
if not cell_metadata_local_path.exists():
    cell_metadata_og_path = cohort_analysis_path / f"asap-{dataset_team}.final_metadata.csv"
    !cp {cell_metadata_og_path} {cell_metadata_local_path}

# load the adata object
cell_metadata_df = pd.read_csv(cell_metadata_local_path, low_memory=False)
print(f"We have loaded the cell_metadata for N={cell_metadata_df.shape[0]} cells")


In [ ]:
# backed="r" leaves the matrix on disk. At this stage we only need .obs,
# and the full object won't fit in the VM's memory.

adata_local_path = local_data_path / f"asap-{dataset_team}.final.h5ad"

# Check if the adata file already exists locally.
if not adata_local_path.exists():
    adata_cell_metadata_og_path = cohort_analysis_path / f"asap-{dataset_team}.final.h5ad"
    !cp {adata_cell_metadata_og_path} {adata_local_path}

adata = sc.read_h5ad(adata_local_path, backed="r")
adata

## 2. Data Preparation

### 2.1 Merge Metadata Types
With both metadata tables and anndata loaded, we can begin prepare the dataset for exploration. A key step is **merging dataset‑level metadata into cell‑level metadata**. This allows us to annotate each cell with experimental conditions and subject information, enabling richer analyses.

Specifically, we combine:
- **Sample‑level metadata** (`SAMPLE.csv`)  
- **Subject‑level metadata** (`SUBJECT.csv`)  
- **Clinical pathology metadata** (`CLINPATH.csv`)  

From each table, we select only the relevant columns (IDs, demographics, brain regions, conditions) to keep the merged metadata concise and focused. This merged metadata will later allow us to subset the dataset (e.g., by diagnosis or brain region) and encode Parkinson’s disease state for downstream analysis.


#### Load Metadata Tables & Select Columns

To keep the metadata compact and analysis-ready, we select only the fields needed for:

- identifying samples and subjects
- demographic variables
- brain region information
- condition or diagnosis

In [ ]:
# asap-crn

SAMPLE = pd.read_csv(ds_metadata_path / "SAMPLE.csv", low_memory=False)
SUBJECT = pd.read_csv(ds_metadata_path / "SUBJECT.csv", low_memory=False)
CLINPATH = pd.read_csv(ds_metadata_path / "CLINPATH.csv", low_memory=False)

source_tables = {
    "SAMPLE" : SAMPLE,
    "SUBJECT": SUBJECT,
    "CLINPATH": CLINPATH,
}


print("SAMPLE:", SAMPLE.shape)
print("SUBJECT:", SUBJECT.shape)
print("CLINPATH:", CLINPATH.shape)

In [ ]:
# define merge keys
subject_keys = [
    "ASAP_subject_id",
    "ASAP_dataset_id",
    "ASAP_team_id",
    "subject_id",
]

clinpath_keys = [
    "ASAP_team_id",
    "ASAP_dataset_id",
    "ASAP_subject_id",
    "subject_id",
    "source_subject_id",
]

#### Clean Metadata Tables

We now clean the metadata tables, ensuring there is no duplicate keys as we merge.
The goal is to construct a single sample-level dataframe (df) that captures all relevant attributes. 

In [ ]:
def collapse_duplicate_records(data, keys, table_name):
    """Reduce a table to one row per key, keeping the most complete record.

    Some subjects appear more than once with partially filled rows. For each
    group we keep the row with the most non-null fields, fill remaining gaps
    from its siblings, and record any field where the rows disagree.

    Returns (collapsed, conflicts, ties) -- always look at `conflicts`, since
    a non-empty result means two sources reported different values for the
    same subject.
    """
    work = data.copy()
    work = work.replace({None: pd.NA, "": pd.NA})

    metadata_columns = [c for c in work.columns if c not in keys]

    work["_fields_present"] = work[metadata_columns].notna().sum(axis=1)
    work["_source_row"] = np.arange(len(work))

    selected_records = []
    conflict_records = []
    tie_records = []

    group_key = keys[0] if len(keys) == 1 else list(keys)

    for key_values, group in work.groupby(group_key, dropna=False, sort=False):
        if not isinstance(key_values, tuple):
            key_values = (key_values,)

        key_record = dict(zip(keys, key_values))
        max_fields = int(group["_fields_present"].max())

        top_rows = group.loc[group["_fields_present"].eq(max_fields)].sort_values("_source_row")
        selected = top_rows.iloc[0].copy()

        if len(top_rows) > 1:
            tie_records.append({
                **key_record,
                "table": table_name,
                "n_tied_rows": len(top_rows),
                "fields_present": max_fields,
                "source_rows": " | ".join(top_rows["_source_row"].astype(str).tolist()),
            })

        conflicting_fields = []

        for column in metadata_columns:
            values = group[column].dropna().drop_duplicates()

            if len(values) == 1:
                selected[column] = values.iloc[0]

            elif len(values) > 1:
                conflicting_fields.append(column)

                selected_top_values = top_rows[column].dropna()
                if len(selected_top_values):
                    selected[column] = selected_top_values.iloc[0]
                elif pd.isna(selected[column]):
                    selected[column] = values.iloc[0]

                conflict_records.append({
                    **key_record,
                    "table": table_name,
                    "field": column,
                    "n_unique_values": len(values),
                    "observed_values": " | ".join(repr(v) for v in values.tolist()),
                    "selected_value": repr(selected[column]),
                    "n_source_records": len(group),
                })
        selected_records.append(selected)

    collapsed = (
        pd.DataFrame(selected_records)
        .drop(columns=["_fields_present", "_source_row"])
        .reset_index(drop=True)
    )
    conflicts = pd.DataFrame(conflict_records)
    ties = pd.DataFrame(tie_records)

    if collapsed.duplicated(list(keys)).any():
        raise AssertionError(f"{table_name} cleaning did not produce one row per key.")

    return collapsed, conflicts, ties


In [ ]:
# ensure no duplicated subject_ids
SUBJECT_clean, SUBJECT_conflicts, SUBJECT_selection_ties = collapse_duplicate_records(
    SUBJECT,
    subject_keys,
    table_name="SUBJECT"
)

print("Original SUBJECT rows:", len(SUBJECT))
print("Clean SUBJECT rows:", len(SUBJECT_clean))
print("SUBJECT conflicts:", len(SUBJECT_conflicts))
print("SUBJECT top-record ties:", len(SUBJECT_selection_ties))

display(SUBJECT_conflicts)

In [ ]:
expected_sample_rows = len(SAMPLE)

df = pd.merge(
    SAMPLE,
    SUBJECT_clean,
    on=subject_keys,
    how="left",
    validate="many_to_one",
)

assert len(df) == expected_sample_rows

In [ ]:
clinpath_duplicate_counts = (
    CLINPATH.groupby(clinpath_keys, dropna=False)
    .size()
    .reset_index(name="n_clinpath_rows")
    .query("n_clinpath_rows > 1")
    .sort_values("n_clinpath_rows", ascending=False)
)


CLINPATH_clean, CLINPATH_conflicts, CLINPATH_selection_ties = collapse_duplicate_records(
    CLINPATH,
    clinpath_keys,
    table_name="CLINPATH",
)

print("Original CLINPATH rows:", len(CLINPATH))
print("Clean CLINPATH rows:", len(CLINPATH_clean))
print("CLINPATH conflicts:", len(CLINPATH_conflicts))
print("CLINPATH top-record ties:", len(CLINPATH_selection_ties))


In [ ]:
before_merge = len(df)

df = pd.merge(
    df,
    CLINPATH_clean,
    on=clinpath_keys,
    how="left",
    validate="many_to_one"
)

assert len(df) == before_merge

# create unique sample identifier
df["sample"] = df["ASAP_sample_id"] + "_" + df["replicate"]


#### Harmonize Brain Region Labels

Map region_level_2 brain regions to a coaser taxonomy. 

In [ ]:
# Region fields arrive as "Name (ABBREV, UBERON:0000000)".
# Split them apart so we can filter on the plain name.

for i in [1, 2, 3]:
    col = f"region_level_{i}"
    df[[f"{col}_name", f"{col}_abbrev", f"{col}_ontology_term_id"]] = (
        df[col].str.extract(r"^(.*?) \((.*?), (.*?)\)$")
    )


In [ ]:
print(df.region_level_1_name.value_counts(), 
      df.region_level_2_name.value_counts(), 
      df.region_level_3_name.value_counts())

In [ ]:
df["region_level_2_name"] = df["region_level_2_name"].str.strip()

brain_simple = {
    "Prefrontal cortex":        "frontal_ctx",
    "Middle frontal gyrus":     "frontal_ctx",
    "Anterior cingulate gyrus": "cingulate_ctx",
    "Inferior parietal lobule": "parietal_ctx",
    "Middle temporal gyrus":    "temporal_ctx",
    "Hippocampus":              "subcortical",
    "Amygdala":                 "subcortical",
    "Putamen":                  "subcortical",
    "Substantia nigra":         "subcortical",
}

df["brain_region_simple"] = df["region_level_2_name"].map(brain_simple)

# Two different problems, so check them separately:
# a labeled region we forgot is a bug; a missing label is a data gap.
unknown = set(df["region_level_2_name"].dropna()) - set(brain_simple)
assert not unknown, f"Add to brain_simple: {unknown}"

n_missing = df["region_level_2_name"].isna().sum()
print(f"{n_missing} samples have no region label and will be excluded.")

In [ ]:
# df has one row per sample; adata.obs has 3.6M rows. These dicts are the bridge to annotate the cells in our adata.

assert df["sample"].is_unique, "sample IDs collapsed silently — check replicate suffix"

# Define sample to match
region_2_mapper_full = dict(zip(df["sample"], df["region_level_2_name"]))
region_2_mapper_simple = dict(zip(df["sample"], df["region_level_2_name"].map(brain_simple)))

# Parkinsons and control samples
condition_id_mapper = dict(zip(df["sample"], df["condition_id"]))
## case_id comes from gp2_phenotype, the harmonized GP2 label
case_id_mapper = dict(zip(df["sample"], df["gp2_phenotype"]))

# Detailed brain region mapper 
region_1_mapper = dict(zip(df["sample"], df["region_level_1_name"]))
region_3_mapper = dict(zip(df["sample"], df["region_level_3_name"]))

# Diagnoses
diagnoses_mapper = dict(zip(df["sample"], df["primary_diagnosis"]))

In [ ]:
#drop all empty columns
# Merging three wide tables leaves columns that are null for every sample.
df = df.convert_dtypes().dropna(axis=1, how="all")
df.head(5)

#### Save Merged Metadata

Now that the dataset-level metadata is assembled, we save it for later use.

In [ ]:
dataset_metadata_filen = local_data_path / "asap-cohort-dataset-metadata.csv"
df.to_csv(dataset_metadata_filen)

#### Add metadata to our cohort data (`adata.obs`)

In [ ]:
# Map samples to metadata
adata.obs["region_level_2"] = adata.obs["sample"].map(region_2_mapper_full)
adata.obs["brain_region_simple"] = adata.obs["sample"].map(region_2_mapper_simple)
adata.obs["case_id"] = adata.obs["sample"].map(case_id_mapper)
adata.obs["condition_id"] = adata.obs["sample"].map(condition_id_mapper)
adata.obs["region_level_1"] = adata.obs["sample"].map(region_1_mapper)
adata.obs["region_level_3"] = adata.obs["sample"].map(region_3_mapper)
adata.obs["primary_diagnosis"] = adata.obs["sample"].map(diagnoses_mapper)

### 2.2 QC Overview Before Subsetting

Before we subset the data, we take a high-level look at how cells are distributed across a 200k subsample of the asap-cohort dataset.These summaries help us understand overall dataset balance, sampling depth, and the representation of conditions and brain regions.

In [ ]:
qc_cols = ["pct_counts_rb", "pct_counts_mt", "doublet_score", "n_genes_by_counts"]

# 200k points is visually indistinguishable from 3.6M here — the plot is fully
# overplotted well before that. Shuffle so no single sample lands on top.
N = 200_000
plot_df = cell_metadata_df.sample(min(N, len(cell_metadata_df)), random_state=0)

xy = plot_df[["UMAP_1", "UMAP_2"]].to_numpy()

fig, axes = plt.subplots(2, 2, figsize=(11, 9))

for ax, col in zip(axes.flat, qc_cols):
    v = plot_df[col].to_numpy()
    # Clip to the 1st–99th percentile: a handful of extreme cells otherwise
    # compress the colormap until every real difference looks flat.
    lo, hi = np.nanpercentile(v, [1, 99])
    pts = ax.scatter(
        xy[:, 0], xy[:, 1], c=v, s=2, alpha=0.5,
        cmap="viridis", vmin=lo, vmax=hi,
        linewidths=0, rasterized=True,
    )
    fig.colorbar(pts, ax=ax, label=col)
    ax.set(title=col, xlabel="UMAP_1", ylabel="UMAP_2")

fig.tight_layout()
plt.show()

In [ ]:
adata.obs["condition_id"].value_counts()

In [ ]:
adata.obs["region_level_2"].value_counts()

### 2.3 Subsetting the Dataset

The full `asap-cohort` **PMDBS snRNA-seq dataset** contains gene expression measurements for **millions of cells** across **tens of thousands of genes**. For efficient analysis and clearer demonstrations in this workshop, we will focus on a biologically relevant subset: **Prefrontal Cortex (PFC) cells.**

This reduced dataset retains:
- the **full gene expression matrix** for these cells,
- all **metadata** fields
  
  
> **Why Subset?**
> It reduces memory and compute requirements while still supporting PD-relevant comparisons (e.g., case vs. control).


> **Alternate strategy:**
> You could load each contributing dataset individually and analyze them separately, which is less resource-intensive. However, for this workshop we leverage the shared cohort latent space to enable consistent UMAP visualization and cross-dataset comparisons.


#### Subsetting by Brain Region and Case/Control Status

We begin by identifying all cells annotated as originating from prefrontal cortex samples, then exclude ambiguous “Other” case labels (i.e Prodromal). 

This prepares the dataset for:
- pseudo-bulk differential expression
- cross-dataset meta-analysis
- UMAP visualization in the shared cohort embedding

In [ ]:
# Identify frontal cortex cells
frontal_cells = adata.obs["brain_region_simple"] == "frontal_ctx"

# Exclude samples with ambiguous case labels
case_control_cells = adata.obs["case_id"].isin(["PD", "Control"])

# Final boolean mask for subsetting
include = frontal_cells & case_control_cells

#### Create the Prefrontal Cortex Subset

With our filters defined, we can now extract the frontal cortex case/control cells into a new **in-memory** `AnnData` object.

> **Note:**  
> The full cohort dataset is read in *backed* mode, meaning it is **not** loaded into memory.  
> Here we extract the prefrontal cortex cells **into memory** and then close the backed file.



In [ ]:
frontal_ad = adata[include].to_memory()
adata.file.close()  # close the original adata file

The resulting subset will serve as the basis for our downstream analyses in this workshop.

Let's take a look at the subset composition

In [ ]:
frontal_ad.obs["condition_id"].value_counts()

In [ ]:
# UMAP - Case/Control
sc.pl.embedding(frontal_ad, basis="umap", color=["condition_id"])

#### Exporting the Subset

Let's save the prefrontal cortex case/control subset as a standalone `AnnData` object.  This gives us a lightweight artifact that preserves embeddings, annotations, and cohort-level integration results.


In [ ]:
frontal_samples_filename = (
    local_data_path / f"asap-{dataset_team}.frontal_ctx_case_control_samples.h5ad"
)
frontal_ad.write_h5ad(frontal_samples_filename)

#remove files from memory
del adata, frontal_ad
gc.collect()

### 2.4 Load Full Gene Expression for the Subset

The subset we created so far contains **only highly variable genes**, which is ideal for integration and visualization but not always sufficient for:
- gene-level exploration  
- marker discovery  
- differential expression

To recover **full gene expression** for the same cells, we:

1. Load the **full, unfiltered cohort AnnData** (backed on disk).  
2. Reload the **subset AnnData** that defines the exact cell set of interest.  
3. Use the subset’s cell index (`obs_names`) to extract the corresponding rows from the full matrix.  

This gives us an `X` matrix with full gene coverage for the PFC case/control cells, while still leveraging the integrated latent space from the cohort workflow.


In [ ]:
# Define file paths 
full_adata_filename = (
    cohort_analysis_path / f"asap-{dataset_team}.merged_cleaned_unfiltered.h5ad"
)
l_full_adata_filename = (
    local_data_path / f"asap-{dataset_team}.merged_cleaned_unfiltered.h5ad"
)

if not l_full_adata_filename.exists():
    !cp {full_adata_filename} {l_full_adata_filename}

In [ ]:
# Load full expression matrix
full_adata = sc.read_h5ad(l_full_adata_filename, backed="r")

In [ ]:
# Reload PFC cell subset
frontal_ad = sc.read_h5ad(frontal_samples_filename, backed="r")

# Extract and select PFC cells from complete gene expression matrix
var_ = full_adata.var.copy()
X = full_adata[frontal_ad.obs_names].X.copy()

full_adata.file.close()

Now we can combine the _full_ gene expression matrix with our frontal cortex subset, and save the resulting `AnnData`object.

In [ ]:
frontal_full_ad = sc.AnnData(
    X=X,
    obs=frontal_ad.obs,
    var=var_,
    uns=frontal_ad.uns,
    obsm=frontal_ad.obsm,
)

In [ ]:
# Save full frontal cortex Anndata object
frontal_full_samples_filename = (
    local_data_path / f"asap-{dataset_team}.full_frontal_ctx_case_control_samples.h5ad"
)
frontal_full_ad.write_h5ad(frontal_full_samples_filename)

## 4. Data Exploration

With our subset prepared, we can now explore the dataset to understand its structure, quality, and biological composition.
This section walks through:
1. Sample composition
2. Cell composition
3. QC metric distributions
4. Integrated embeddings (UMAP)
5. Gene expression

These steps help build an intuitive overview of the dataset before moving into deeper analyses.

### 4.1 Sample Composition

First, let's summarize the number of samples contributing to this subset and how they break down across metadata categories such as:
- Case vs Control
- Diagnosis
- Brain region (frontal cortex only, but with fine-grained sublabels if available)

This provides context for downstream PD analyses and highlights any imbalances between groups.

In [ ]:
obs = frontal_full_ad.obs

# One row per sample. Everything in this section is study design, so counting
# cells here would just report sequencing depth instead.
sample_meta = obs.drop_duplicates("sample").set_index("sample")

print(f"Samples: {len(sample_meta)}")
print(f"Cells:   {frontal_full_ad.n_obs:,}")
print(f"\nSamples per condition:\n{sample_meta['condition_id'].value_counts().to_string()}")

# Diagnosis is the contributing team's own wording; condition_id is the
# harmonized label. Crossing them shows how the free-text labels were grouped.
print("\nSamples by condition and diagnosis:")
display(pd.crosstab(sample_meta["condition_id"], sample_meta["primary_diagnosis"]))

# frontal_ctx pools two region labels. If one dominates a single arm, an
# apparent case/control effect could really be a region effect.
print("\nSamples by region and condition:")
display(pd.crosstab(sample_meta["region_level_2"], sample_meta["condition_id"]))

### 4.2 Cell Composition

Next, we examine how cells are distributed across metadata categories. This helps answer questions like:
- How many cells were captured per sample?
- Are there differences in cell counts between case vs control?
- Are any samples under- or over-represented?

In [ ]:
# Count cells per sample
cells_per_sample = frontal_full_ad.obs["sample"].value_counts()

plt.figure(figsize=(9,4))
sns.barplot(x=cells_per_sample.index, y=cells_per_sample.values)
plt.title("Number of Cells Captured per Sample")
plt.xlabel("Sample ID")
plt.ylabel("Number of Cells")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Create a contingency table
ct = pd.crosstab(frontal_full_ad.obs["condition_id"], frontal_ad.obs["primary_diagnosis"])

# Plot stacked bar chart
ct.plot(kind="bar", stacked=True, figsize=(6,3))
plt.title("Cell Counts by Case/Control Status (stacked by Diagnosis)")
plt.xlabel("Case/Control")
plt.ylabel("Number of Cells")
plt.xticks(rotation=45)
plt.legend(title="Diagnosis", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
cell_counts = frontal_full_ad.obs["cell_type"].value_counts()

plt.figure(figsize=(4,2))
sns.barplot(x=cell_counts.index, y=cell_counts.values)
plt.xticks(rotation=90)
plt.title("Cell Counts per Cell Type,")
plt.ylabel("Number of Cells")
plt.show()


### 4.3 QC Distributions by Group

We now examine QC metrics across biologically relevant groups.
                      
**Common QC metrics include:**
- n_genes_by_counts
- total_counts
- pct_counts_mt
- pct_counts_rb

In [ ]:
qc_metrics = ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_rb"]

# Create a 2x2 grid of QC subplots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, m in zip(axes.flatten(), qc_metrics):
    sns.violinplot(data=frontal_full_ad.obs, x="condition_id", y=m, cut=0, ax=ax)
    ax.set_title(f"{m} by Case/Control Status")
    ax.set_xlabel("Case/Control")
    ax.set_ylabel(m)
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


### 4.4 Explore Integrated Embeddings (UMAP)

Finally, we visualize the prefrontal cortex cells in the UMAP embedding.
This reveals overall structure of:

- major cell populations
- batch or dataset mixing
- case vs control separation (if present)

In [ ]:
sc.pl.umap(
    frontal_full_ad,
    color=["region_level_2", "brain_region_simple","batch_id",
           "condition_id", "primary_diagnosis", 
           "cell_type", "phase",
           "n_genes_by_counts", "pct_counts_mt"],
    ncols=2,
    wspace=0.4,
    s=8,
)

### 4.5 Explore Gene Expression Profiles

Understanding how gene expression varies across cell types and disease-relevant pathways is a critical early step before downstream analysis. Here we visualize expression of PD-relevant genes across annotated cell types using the `cell_type` labels from the curated cohort.

The genes below include lysosomal, synaptic, inflammatory, and neurodegenerative-related markers reported in PD literature.

In [ ]:
pfc_genes = [
    "SNCA", "MAPT", "UCHL1", "LRRK2", "CTSB", "BAG3", "RIT2"
]

# Find which of these genes are present in your dataset
genes_present = [g for g in pfc_genes if g in frontal_full_ad.var_names]

print("Genes found:", genes_present)


#### Let's visualize expression across the integrated UMAP

In [ ]:
sc.pl.umap(frontal_full_ad, 
           color=genes_present, 
           cmap=sns.cubehelix_palette(dark=0, light=0.9, as_cmap=True),
          ncols = 3)

In [ ]:
sc.pl.dotplot(
    frontal_full_ad,
    var_names=genes_present,
    groupby="cell_type",
    standard_scale="var",
    figsize=(6, 4),
)

## 5. Provenance

This notebook was generated as part of the **ASAP-CRN Verily Workbench Learning Lab** to demonstrate exploratory analysis workflows using harmonized single-nucleus RNA-seq data from the ASAP Collaborative Research Network.

### **Software & Environment**
- Platform: **Verily Workbench**
- Runtime: Cloud app environment - Jupyter Lab (Python)
- Key libraries: `scanpy`, `anndata`, `pandas`, `numpy`, `matplotlib`, `seaborn`
- Data format: `AnnData` (`.h5ad`)

### **Source Data**
The dataset analyzed here originates from the integrated ASAP CRN post-mortem brain cohort:

- Curation Workflow: **PMDBS scRNAseq Pipeline**  
- Data type: **cohort-level harmonized single-nucleus RNA-seq**
- Input samples: Multiple contributing datasets aggregated into a shared latent space
- Data storage:
  - Harmonized cohort files: `workspace/01_PMDBS/pmdbs-sc-rnaseq-v3/cohort-pmdbs-sc-rnaseq/pmdbs_sc_rnaseq/cohort_analysis`

Full metadata tables used:
- `SAMPLE.csv` — sample-level descriptors
- `SUBJECT.csv` — donor/demographic attributes
- `CLINPATH.csv` — clinical pathology annotations

### **Notebook-Generated Outputs**
This notebook produces intermediate artifacts stored locally under: `ws_files/pilot_workshop_files/`

In [ ]:
!date

In [ ]:
!jupyter labextension list

In [ ]:
print("Number of cores:")
!grep ^processor /proc/cpuinfo | wc -l

In [ ]:
print("Memory") 
!grep "^MemTotal:" /proc/meminfo

## 6. Next Steps

In this notebook, you:
- Integrated subject, sample, and condition metadata
- Subset to prefrontal cortex case/control cells
- Explored dataset composition and embeddings
- Highlighted PD-relevant gene expression patterns

Next, we shift from exploration → downstream analysis.

👉 Continue to the next notebook:  **[03_downstream_analysis.ipynb](./03_downstream_analysis.ipynb)**.
Here we’ll perform differential expression, pathway analysis, and cell-type–specific comparisons.
